# PPE Monitor — Quick Start

Just three cells:

| Cell | What it does | Time |
|---|---|---|
| 1 | Connect Drive + clone the repo (code & weights) + load models | **First run:** ~10 min (TensorRT engine build, cached on Drive)<br>**Later runs:** ~1 min |
| 2 | Analyze a video or image — just give it a path | — |
| 3 | (Optional) upload a file from your computer and analyze it | — |

**Drive cache:** `MyDrive/ppe_monitor/`

```text
ppe_monitor/
├── weights/            ← .pt weights (seeded from the repo, automatic)
├── engines/Tesla_T4/   ← TensorRT engines (automatic, one folder per GPU type)
├── inputs/             ← put your videos and images here
└── outputs/            ← results + events/ (violation snapshots)
```

Do not forget: Runtime → Change runtime type → **T4 GPU**.

In [ ]:
# ============================================================
# 1) Setup — once per session
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!git clone --depth 1 https://github.com/ParsaVictor/PPE_detection-Sentinel /content/sentinel
!pip -q install ultralytics "lap>=0.5.12" arabic-reshaper python-bidi

import os, shutil, sys
BASE = "/content/drive/MyDrive/ppe_monitor"
os.makedirs(BASE, exist_ok=True)

# Seed the weight cache from the repository copy (skips every external download)
shutil.copytree("/content/sentinel/weights", os.path.join(BASE, "weights"), dirs_exist_ok=True)

sys.path.insert(0, "/content/sentinel")
import ppe_monitor as pm
pm.init()   # first run also builds the TensorRT engines; later runs load them

In [ ]:
# ============================================================
# 2) Analyze — just give a path (bare name = from the Drive inputs/ folder)
# ============================================================
result = pm.analyze("3.mp4")

# More examples:
# pm.analyze("/content/drive/MyDrive/videos/site_cam1.mp4")
# pm.analyze("photo.jpg")
# pm.analyze("3.mp4", debug=True)          # rejected detections with their reasons
# pm.analyze("long.mp4", max_seconds=30)   # only the first 30 seconds
# pm.analyze("3.mp4", benchmark=True)      # + TensorRT vs PyTorch speed/accuracy comparison

In [ ]:
# ============================================================
# 3) (Optional) upload from your computer and analyze
# ============================================================
from google.colab import files
for name, data in files.upload().items():
    path = pm.INPUT_DIR / name
    path.write_bytes(data)
    pm.analyze(path)

## Common settings

```python
pm.cfg.show_skeleton = False      # no skeleton (then run pm.init() again)
pm.cfg.person_conf = 0.15         # more distant people
pm.cfg.on_violation = 2.0         # more conservative alerts
pm.cfg.required_ppe = ("helmet",) # only the helmet is mandatory
```

All parameters and the "which symptom → which parameter" guide live in the repository `README.md`. If TensorRT misbehaves, `pm.init(backend="pytorch")` behaves exactly like the reference V2 pipeline.